In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dataset 05 — Adult (Census Income) — UCI

**Descripción:** Dataset extraído del censo de EE.UU. de 1994 por Barry Becker. Conocido también como "Census Income". Contiene información demográfica y laboral para predecir si el ingreso anual de una persona supera los $50,000.

**Tarea:** Clasificación binaria — predecir si el ingreso es `>50K` o `<=50K`.

**Características:**
- m = 32,561 registros (train) + 16,281 (test)
- n = 14 características (6 continuas + 8 categóricas)
- Variable objetivo: `income` (`<=50K` o `>50K`)
- Valores nulos: codificados como `?` en workclass, occupation, native-country
- Desbalanceado: 75.9% gana ≤$50K, 24.1% gana >$50K

**URL UCI:** https://archive.ics.uci.edu/ml/datasets/adult

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
print('Librerías cargadas correctamente')

## 1. Carga del Dataset

In [ ]:
columnas = ['age','workclass','fnlwgt','education','education-num',
           'marital-status','occupation','relationship','race','sex',
           'capital-gain','capital-loss','hours-per-week','native-country','income']

ruta = '/content/drive/MyDrive/machine learning/datasets/adult.data'
data = pd.read_csv(ruta, names=columnas, na_values=' ?', skipinitialspace=True)

print(f'Dimensiones: {data.shape}')
data.head()

## 2. Revisión de Tipos y Nulos

In [ ]:
print(data.dtypes)
print()
nulos = data.isnull().sum()
print('Valores nulos:')
print(nulos[nulos > 0])
print(f'Total: {data.isnull().sum().sum()}')

## 3. Preprocesamiento con Pandas

In [ ]:
df = data.copy()

# Imputar nulos categóricos con moda
cols_cat = df.select_dtypes(include=['object']).columns
for col in cols_cat:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Codificar variable objetivo
df['income_bin'] = (df['income'].str.strip() == '>50K').astype(int)

# Codificar todas las categóricas
for col in cols_cat:
    df[col] = pd.Categorical(df[col]).codes

print(f'Nulos restantes: {df.isnull().sum().sum()}')
print(f'\nDistribución de ingresos:')
print(f'  <=50K (0): {int((df["income_bin"]==0).sum())} ({(df["income_bin"]==0).mean()*100:.1f}%)')
print(f'  >50K  (1): {int((df["income_bin"]==1).sum())} ({(df["income_bin"]==1).mean()*100:.1f}%)')

## 4. Estadísticas Descriptivas

In [ ]:
df.describe()

## 5. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(data['age'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Distribución de Edad')
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Frecuencia')

axes[1].bar(['<=50K', '>50K'],
            [int((df['income_bin']==0).sum()), int((df['income_bin']==1).sum())],
            color=['steelblue','tomato'], edgecolor='black')
axes[1].set_title('Distribución de Ingreso')
axes[1].set_ylabel('Número de personas')

axes[2].hist(data['hours-per-week'], bins=30, color='green', edgecolor='black')
axes[2].set_title('Horas Trabajadas por Semana')
axes[2].set_xlabel('Horas/semana')

plt.tight_layout()
plt.show()

## 6. División 80/20

In [ ]:
X_cols = [col for col in df.columns if col not in ['income','income_bin']]
X = df[X_cols].to_numpy(dtype=np.float64)
y = df['income_bin'].to_numpy(dtype=np.float64)

np.random.seed(42)
idx = np.random.permutation(len(X))
corte = int(len(X)*0.8)
X_train,X_test = X[idx[:corte]],X[idx[corte:]]
y_train,y_test = y[idx[:corte]],y[idx[corte:]]
print(f'Entrenamiento: {X_train.shape[0]} | Prueba: {X_test.shape[0]}')

## 7. Conclusiones

**Adult (Census Income)** es un clásico dataset de clasificación binaria con 32,561 registros y 14 características demográficas y laborales. Los valores nulos están codificados como `?` en workclass, occupation y native-country (≈5-6% de los datos). El dataset está desbalanceado: el 75.9% gana ≤$50K. Es ideal para aplicar regresión logística para clasificación binaria. Las variables más predictivas suelen ser education-num, hours-per-week, age y marital-status.